# 01 — GAN 2D Normalized
**Dự án:** Latent Manipulation of Brain MRI using Volume-Preserving GANs

**Input:** PNG normalized từ `00a_preprocessing_2d.ipynb`

**Kiến trúc:**
- **Generator**: U-Net + Age Embedding inject vào bottleneck
- **Discriminator**: PatchGAN + Age Conditioning

**Output:**
```
gan2d_normalized/
└── best_model.pth   ← checkpoint tốt nhất (loss_G thấp nhất)
```

## Bước 1: Cấu hình

In [1]:
import os

# ==== ĐƯỜNG DẪN ====
DATA_DIR   = '/kaggle/input/datasets/minhbodoi/pre2dthatlan2/preprocessed_2d/normalized'
LABELS_CSV = '/kaggle/input/datasets/minhbodoi/pre2dthatlan2/preprocessed_2d/preprocessing_log.csv'
OUTPUT_DIR = '/kaggle/working/gan2d_normalized'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==== HYPERPARAMETERS ====
IMAGE_SIZE = 256
BATCH_SIZE = 16
NUM_EPOCHS = 300
LR_G       = 2e-4
LR_D       = 2e-4
LAMBDA_L1  = 100
LAMBDA_AGE = 1
LATENT_DIM = 256

print(f'Data dir : {DATA_DIR}')
print(f'PNG files: {len([f for f in os.listdir(DATA_DIR) if f.endswith(".png")])}')

Data dir : /kaggle/input/datasets/minhbodoi/pre2dthatlan2/preprocessed_2d/normalized
PNG files: 299


## Bước 2: Import thư viện

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cuda
GPU : Tesla T4
VRAM: 15.6 GB


## Bước 3: Dataset

In [3]:
class BrainMRI2DDataset(Dataset):
    def __init__(self, data_dir, labels_csv, image_size=256):
        self.data_dir = data_dir
        df = pd.read_csv(labels_csv)
        df['full_path'] = df['png_file'].apply(lambda x: os.path.join(data_dir, x))
        df = df[df['full_path'].apply(os.path.exists)].reset_index(drop=True)
        self.df      = df
        self.age_min = df['age'].min()
        self.age_max = df['age'].max()
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5])
        ])
        print(f'Dataset: {len(df)} ảnh | tuổi [{self.age_min:.1f}, {self.age_max:.1f}]')

    def normalize_age(self, age):
        return 2 * (age - self.age_min) / (self.age_max - self.age_min) - 1

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img      = Image.open(row['full_path']).convert('L')
        img      = self.transform(img)
        age_norm = torch.tensor(self.normalize_age(row['age']), dtype=torch.float32)
        age_raw  = torch.tensor(row['age'] / 100.0, dtype=torch.float32)
        return img, age_norm, age_raw


dataset    = BrainMRI2DDataset(DATA_DIR, LABELS_CSV, IMAGE_SIZE)
dataloader = DataLoader(
    dataset, batch_size=BATCH_SIZE,
    shuffle=True, num_workers=4, pin_memory=True
)

Dataset: 299 ảnh | tuổi [18.0, 64.0]


## Bước 4: Kiến trúc Model

In [4]:
class AgeEmbedding(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 128), nn.ReLU(),
            nn.Linear(128, embed_dim)
        )
    def forward(self, age):
        return self.net(age.unsqueeze(-1))


class UNetBlock(nn.Module):
    def __init__(self, in_ch, out_ch, down=True, use_bn=True, dropout=False):
        super().__init__()
        layers = []
        if down : layers.append(nn.Conv2d(in_ch, out_ch, 4, 2, 1, bias=False))
        else    : layers.append(nn.ConvTranspose2d(in_ch, out_ch, 4, 2, 1, bias=False))
        if use_bn  : layers.append(nn.BatchNorm2d(out_ch))
        if dropout : layers.append(nn.Dropout(0.5))
        layers.append(nn.LeakyReLU(0.2) if down else nn.ReLU())
        self.block = nn.Sequential(*layers)
    def forward(self, x):
        return self.block(x)


class Generator(nn.Module):
    """
    U-Net Generator với age conditioning.
    Input : ảnh (B, 1, 256, 256) + tuổi normalize (B,)
    Output: ảnh sinh ra (B, 1, 256, 256)
    """
    def __init__(self, latent_dim=256):
        super().__init__()
        self.age_embed = AgeEmbedding(latent_dim)
        self.age_proj  = nn.Linear(latent_dim, 512)
        self.e1 = UNetBlock(1,   64,  down=True, use_bn=False)
        self.e2 = UNetBlock(64,  128, down=True)
        self.e3 = UNetBlock(128, 256, down=True)
        self.e4 = UNetBlock(256, 512, down=True)
        self.e5 = UNetBlock(512, 512, down=True)
        self.e6 = UNetBlock(512, 512, down=True)
        self.e7 = UNetBlock(512, 512, down=True)
        self.e8 = UNetBlock(512, 512, down=True, use_bn=False)
        self.d1 = UNetBlock(512,  512, down=False, dropout=True)
        self.d2 = UNetBlock(1024, 512, down=False, dropout=True)
        self.d3 = UNetBlock(1024, 512, down=False, dropout=True)
        self.d4 = UNetBlock(1024, 512, down=False)
        self.d5 = UNetBlock(1024, 256, down=False)
        self.d6 = UNetBlock(512,  128, down=False)
        self.d7 = UNetBlock(256,  64,  down=False)
        self.out = nn.Sequential(nn.ConvTranspose2d(128, 1, 4, 2, 1), nn.Tanh())

    def forward(self, x, age):
        e1 = self.e1(x);  e2 = self.e2(e1); e3 = self.e3(e2); e4 = self.e4(e3)
        e5 = self.e5(e4); e6 = self.e6(e5); e7 = self.e7(e6); e8 = self.e8(e7)
        z  = e8 + self.age_proj(self.age_embed(age)).view(-1, 512, 1, 1)
        d1 = self.d1(z)
        d2 = self.d2(torch.cat([d1, e7], dim=1))
        d3 = self.d3(torch.cat([d2, e6], dim=1))
        d4 = self.d4(torch.cat([d3, e5], dim=1))
        d5 = self.d5(torch.cat([d4, e4], dim=1))
        d6 = self.d6(torch.cat([d5, e3], dim=1))
        d7 = self.d7(torch.cat([d6, e2], dim=1))
        return self.out(torch.cat([d7, e1], dim=1))


class Discriminator(nn.Module):
    """
    PatchGAN Discriminator với age conditioning.
    Input : ảnh (B, 1, H, W) + age channel → (B, 2, H, W)
    """
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(2, 64, 4, 2, 1, bias=False), nn.LeakyReLU(0.2),
            nn.Conv2d(64,  128, 4, 2, 1, bias=False), nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.Conv2d(128, 256, 4, 2, 1, bias=False), nn.BatchNorm2d(256), nn.LeakyReLU(0.2),
            nn.Conv2d(256, 512, 4, 1, 1, bias=False), nn.BatchNorm2d(512), nn.LeakyReLU(0.2),
            nn.Conv2d(512, 1, 4, 1, 1)
        )
    def forward(self, img, age):
        age_map = age.view(-1, 1, 1, 1).expand(-1, 1, img.shape[2], img.shape[3])
        return self.model(torch.cat([img, age_map], dim=1))


G = Generator(LATENT_DIM).to(DEVICE)
D = Discriminator().to(DEVICE)
print(f'Generator    : {sum(p.numel() for p in G.parameters())/1e6:.1f}M params')
print(f'Discriminator: {sum(p.numel() for p in D.parameters())/1e6:.1f}M params')

Generator    : 54.6M params
Discriminator: 2.8M params


## Bước 5: Loss + Optimizers

In [5]:
criterion_gan = nn.BCEWithLogitsLoss()
criterion_l1  = nn.L1Loss()
criterion_age = nn.MSELoss()

age_regressor = nn.Sequential(
    nn.AdaptiveAvgPool2d(8), nn.Flatten(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, 1), nn.Sigmoid()
).to(DEVICE)

opt_G = optim.Adam(
    list(G.parameters()) + list(age_regressor.parameters()),
    lr=LR_G, betas=(0.5, 0.999)
)
opt_D = optim.Adam(D.parameters(), lr=LR_D, betas=(0.5, 0.999))

scheduler_G = optim.lr_scheduler.StepLR(opt_G, step_size=50, gamma=0.5)
scheduler_D = optim.lr_scheduler.StepLR(opt_D, step_size=50, gamma=0.5)

with torch.no_grad():
    d_out_shape = D(
        torch.zeros(1, 1, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE),
        torch.zeros(1).to(DEVICE)
    ).shape
print(f'D output shape: {d_out_shape}')
print('Loss + Optimizers sẵn sàng!')

D output shape: torch.Size([1, 1, 30, 30])
Loss + Optimizers sẵn sàng!


## Bước 6: Training

In [6]:
best_loss_G = float('inf')
history     = {'loss_G': [], 'loss_D': [], 'loss_L1': [], 'loss_age': []}

print(f'Bắt đầu training {NUM_EPOCHS} epochs...\n')

for epoch in range(1, NUM_EPOCHS + 1):
    G.train(); D.train()
    epoch_loss_G = epoch_loss_D = epoch_loss_L1 = epoch_loss_age = 0

    for real_imgs, ages_norm, ages_raw in tqdm(dataloader,
                                               desc=f'Epoch {epoch}/{NUM_EPOCHS}',
                                               leave=False):
        real_imgs  = real_imgs.to(DEVICE)
        ages_norm  = ages_norm.to(DEVICE)
        ages_raw   = ages_raw.to(DEVICE)
        B          = real_imgs.size(0)
        real_label = torch.ones(B,  *d_out_shape[1:]).to(DEVICE)
        fake_label = torch.zeros(B, *d_out_shape[1:]).to(DEVICE)

        # Train Discriminator
        opt_D.zero_grad()
        with torch.no_grad():
            fake_imgs = G(real_imgs, ages_norm)
        loss_D = (
            criterion_gan(D(real_imgs, ages_norm), real_label) +
            criterion_gan(D(fake_imgs, ages_norm), fake_label)
        ) * 0.5
        loss_D.backward()
        opt_D.step()

        # Train Generator
        opt_G.zero_grad()
        fake_imgs  = G(real_imgs, ages_norm)
        loss_G_adv = criterion_gan(D(fake_imgs, ages_norm), real_label)
        loss_L1    = criterion_l1(fake_imgs, real_imgs) * LAMBDA_L1
        pred_age   = age_regressor(fake_imgs).squeeze()
        loss_age   = criterion_age(pred_age, ages_raw) * LAMBDA_AGE
        loss_G     = loss_G_adv + loss_L1 + loss_age
        loss_G.backward()
        opt_G.step()

        epoch_loss_G   += loss_G_adv.item()
        epoch_loss_D   += loss_D.item()
        epoch_loss_L1  += loss_L1.item()
        epoch_loss_age += loss_age.item()

    scheduler_G.step()
    scheduler_D.step()

    n = len(dataloader)
    avg_loss_G   = epoch_loss_G   / n
    avg_loss_D   = epoch_loss_D   / n
    avg_loss_L1  = epoch_loss_L1  / n
    avg_loss_age = epoch_loss_age / n

    history['loss_G'].append(avg_loss_G)
    history['loss_D'].append(avg_loss_D)
    history['loss_L1'].append(avg_loss_L1)
    history['loss_age'].append(avg_loss_age)

    print(f'Epoch {epoch:3d}/{NUM_EPOCHS} | '
          f'loss_G={avg_loss_G:.4f} | '
          f'loss_D={avg_loss_D:.4f} | '
          f'loss_L1={avg_loss_L1:.4f} | '
          f'loss_age={avg_loss_age:.4f}')

    # Lưu best model dựa trên loss_G thấp nhất
    if avg_loss_G < best_loss_G:
        best_loss_G = avg_loss_G
        torch.save({
            'epoch'        : epoch,
            'G_state'      : G.state_dict(),
            'D_state'      : D.state_dict(),
            'opt_G'        : opt_G.state_dict(),
            'opt_D'        : opt_D.state_dict(),
            'history'      : history,
            'age_min'      : dataset.age_min,
            'age_max'      : dataset.age_max,
            'best_loss_G'  : best_loss_G,
        }, f'{OUTPUT_DIR}/best_model.pth')
        print(f'  → Best model saved! (loss_G={best_loss_G:.4f})')

print(f'\n=== TRAINING HOÀN THÀNH ===')
print(f'Best loss_G : {best_loss_G:.4f}')
print(f'Model lưu tại: {OUTPUT_DIR}/best_model.pth')

Bắt đầu training 300 epochs...



Epoch 1/300:   0%|          | 0/19 [00:00<?, ?it/s]

Epoch   1/300 | loss_G=1.4378 | loss_D=0.4471 | loss_L1=55.7721 | loss_age=0.0400
  → Best model saved! (loss_G=1.4378)


Epoch 2/300:   0%|          | 0/19 [00:00<?, ?it/s]

Epoch   2/300 | loss_G=3.2775 | loss_D=0.0618 | loss_L1=35.2894 | loss_age=0.0195


...

Epoch 298/300 | loss_G=0.7053 | loss_D=0.6853 | loss_L1=0.8803 | loss_age=0.0051


Epoch 299/300:   0%|          | 0/19 [00:00<?, ?it/s]

Epoch 299/300 | loss_G=0.7053 | loss_D=0.6866 | loss_L1=0.8854 | loss_age=0.0050


Epoch 300/300:   0%|          | 0/19 [00:00<?, ?it/s]

Epoch 300/300 | loss_G=0.7010 | loss_D=0.6854 | loss_L1=0.8820 | loss_age=0.0050

=== TRAINING HOÀN THÀNH ===
Best loss_G : 0.6927
Model lưu tại: /kaggle/working/gan2d_normalized/best_model.pth


In [7]:
# Tự động lưu model thành Kaggle Dataset sau khi train xong
import json, os, subprocess

dataset_name = os.path.basename(OUTPUT_DIR).replace('_', '-')  # gan2d-normalized
kaggle_user  = subprocess.run('kaggle config view', shell=True, 
                              capture_output=True, text=True).stdout
kaggle_user  = [l.split(':')[1].strip() for l in kaggle_user.split('\n') 
                if 'username' in l][0]

with open(f'{OUTPUT_DIR}/dataset-metadata.json', 'w') as f:
    json.dump({
        'title'   : dataset_name,
        'id'      : f'{kaggle_user}/{dataset_name}',
        'licenses': [{'name': 'CC0-1.0'}]
    }, f)

# Tạo mới hoặc update version nếu đã tồn tại
check = subprocess.run(f'kaggle datasets list --user {kaggle_user} --search {dataset_name}',
                       shell=True, capture_output=True, text=True)
if dataset_name in check.stdout:
    !kaggle datasets version -p {OUTPUT_DIR} -m "update"
else:
    !kaggle datasets create -p {OUTPUT_DIR}

print(f'Done! {kaggle_user}/{dataset_name}')

Starting upload for file best_model.pth
100%|█████████████████████████████████████████| 656M/656M [00:05<00:00, 115MB/s]
Upload successful: best_model.pth (656MB)
Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/minhbodoi/gan2d-normalized
Done! minhbodoi/gan2d-normalized
